# Downstream evaluation: pretraining versus SFT


Select **Runtime → Change runtime type → A100 GPU**. Run cells in order.
All artifacts live under `/content/drive/MyDrive/YLM_runs/<RUN_NAME>`.
Fixed names intentionally resume; change the name for a new experiment. Completed
`metrics.json` arms are skipped. Do not change budgets/model/seeds while resuming.

`src/profiling.py` records throughput, phase totals and per-step times, peak CUDA
memory, GPU utilization, a bounded `torch.profiler` trace, and `summary.json`.
GPU telemetry can be unavailable; profiles cover the script's profiled region,
not setup/download time. Reruns overwrite the individual invocation's profile.

**Time estimates (not measured on Colab):** setup/public downloads may take 5–30 minutes. Full CALiSol/RedDB feature extraction, CPU fingerprint RF fits and bootstrap evaluation across 12 checkpoints can take roughly 1–12 hours; MP API requests can add substantial time. Use `LIMIT` for a separately named pilot. This notebook uses frozen probes (`--probe_only`) because script 22 has no `--resume` or persistent fine-tuning checkpoints. It restarts interrupted task/arm units; completed units are skipped via an atomic `metrics.json` marker. No resumable downstream fine-tuning is claimed. Script 22 itself fits fingerprint ridge and RF baselines for molecular tasks. MP is optional and has no molecular fingerprint baseline.

In [ ]:
RUN_NAME = 'ylm_lm_downstream_eval' #@param {type:"string"}
PRETRAIN_RUN_NAME = 'ylm_lm_pretrain_ablation' #@param {type:"string"}
POSTTRAIN_RUN_NAME = 'ylm_lm_posttrain' #@param {type:"string"}
CHECKPOINT_SEEDS = [0]
EVAL_SEEDS = [0]
ARMS = ['smiles', 'smiles_xyz', 'prefixA', 'xattnA', 'xattnB', 'xattnC']
TASKS = ['calisol', 'reddb', 'mp_battery']
LIMIT = None  # optional integer for a quick subset; None uses the full public data
BATCH_SIZE = 2
MAX_LEN = 512
BOOTSTRAP = 1000
INSTALL_MP_API = False #@param {type:"boolean"}
SETTINGS = dict(pretrain_run=PRETRAIN_RUN_NAME, posttrain_run=POSTTRAIN_RUN_NAME,
                checkpoint_seeds=CHECKPOINT_SEEDS, eval_seeds=EVAL_SEEDS, arms=ARMS,
                tasks=TASKS, limit=LIMIT, batch_size=BATCH_SIZE, max_len=MAX_LEN,
                bootstrap=BOOTSTRAP, probe_only=True, mp_api=INSTALL_MP_API)

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, shutil, urllib.request, zipfile

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/YLM_runs')
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME
SAVE = BASE / RUN_NAME
SAVE.mkdir(parents=True, exist_ok=True)
ROOT = Path('/content/YLM')
BRANCH = 'feat/llm-irreps-ablation-oct30'
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/yaniguan/YLM.git', str(ROOT)], check=True)
subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
os.chdir(ROOT)
os.environ.update(OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false')
packages = ['e3nn==0.6.0', 'numpy<2.3', 'scipy', 'rdkit', 'tblite',
            'transformers>=4.40,<5', 'accelerate', 'scikit-learn', 'pynvml', 'pytest']
if INSTALL_MP_API:
    packages.append('mp-api')
subprocess.run([sys.executable, '-m', 'pip', 'install', *packages], check=True)
import torch
torch.set_num_threads(2)
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print('GPU:', torch.cuda.get_device_name(), '| output:', SAVE)
if 'A100' not in torch.cuda.get_device_name():
    print('Warning: estimates assume A100; this runtime has a different GPU.')

def run_script(script, *args):
    command = [sys.executable, str(ROOT / 'scripts' / script), *map(str, args)]
    print(' '.join(command), flush=True)
    with (SAVE / 'commands.log').open('a') as log:
        log.write(' '.join(command) + '\n')
    subprocess.run(command, cwd=ROOT, check=True)

# Preserve experiment settings on reruns; use a new RUN_NAME for changed settings.
settings = dict(SETTINGS, branch=BRANCH)
manifest = SAVE / 'notebook_config.json'
if manifest.exists():
    assert json.loads(manifest.read_text()) == settings, 'Settings changed: choose a new RUN_NAME.'
else:
    manifest.write_text(json.dumps(settings, indent=2))
(SAVE / 'git_commit.txt').write_text(subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True))

In [ ]:
# Persist public source files, provenance and the optional MP cache on Drive.
public = SAVE / 'data' / 'electrolyte'
public.mkdir(parents=True, exist_ok=True)
local = ROOT / 'data' / 'electrolyte'
local.parent.mkdir(exist_ok=True)
if local.is_symlink():
    if local.resolve() != public.resolve():
        local.unlink()
        local.symlink_to(public, target_is_directory=True)
elif local.exists():
    shutil.copytree(local, public, dirs_exist_ok=True)
    shutil.rmtree(local)
    local.symlink_to(public, target_is_directory=True)
else:
    local.symlink_to(public, target_is_directory=True)
run_script('download_public_data.py', '--out', public)

In [ ]:
# Optional Colab secret; never print or save the key.
if INSTALL_MP_API:
    try:
        from google.colab import userdata
        key = userdata.get('MP_API_KEY')
        if key:
            os.environ['MP_API_KEY'] = key
        del key
    except Exception:
        print('MP_API_KEY unavailable; MP will be explicitly skipped.')
else:
    os.environ.pop('MP_API_KEY', None)
    print('Optional MP disabled; enable INSTALL_MP_API and add a Colab MP_API_KEY secret.')

In [ ]:
PRETRAIN = BASE / PRETRAIN_RUN_NAME
POSTTRAIN = BASE / POSTTRAIN_RUN_NAME
for stage, parent in [('pretrain', PRETRAIN), ('sft', POSTTRAIN)]:
    for checkpoint_seed in CHECKPOINT_SEEDS:
        for arm in ARMS:
            source = parent / stage / f'{arm}_seed{checkpoint_seed}'
            assert (source / 'metrics.json').exists(), f'Missing completed checkpoint: {source}'
            for task in TASKS:
                name = f'{stage}_{arm}_seed{checkpoint_seed}_{task}'
                out = SAVE / 'downstream' / name
                if (out / 'metrics.json').exists():
                    print('Skip completed:', name)
                    continue
                run_script('22_downstream_eval.py', '--runs', source, '--tasks', task,
                           '--seeds', *EVAL_SEEDS, '--probe_only', '--batch_size', BATCH_SIZE,
                           '--max_len', MAX_LEN, '--bootstrap', BOOTSTRAP, '--device', 'cuda',
                           '--out', out, '--profile_dir', SAVE / 'profiling' / name,
                           *([] if LIMIT is None else ['--limit', LIMIT]))
                # Script 22 writes results.json, not metrics.json. Publish an atomic
                # completion marker only after success; interrupted jobs rerun this unit.
                result = json.loads((out / 'results.json').read_text())
                skipped = [r for r in result['records'] if 'skipped' in r]
                # Missing MP credentials must remain retryable when enabled later.
                if not (task == 'mp_battery' and any('No rows' in r['skipped'] for r in skipped)):
                    temporary = out / 'metrics.tmp'
                    temporary.write_text(json.dumps(result, indent=2))
                    temporary.replace(out / 'metrics.json')

In [ ]:
lines = ['# Downstream frozen probes and fingerprint baselines', '',
         '| Stage | Checkpoint seed | Task | Target | Arm | Method | Eval seeds | MAE | RMSE | R2 | Spearman |',
         '|---|---|---|---|---|---|---|---|---|---|---|']
def number(value):
    return 'N/A' if value is None else f'{value:.5g}'
for path in sorted((SAVE / 'downstream').glob('*/results.json')):
    result = json.loads(path.read_text())
    # Interrupted units can have a partial results.json; omit until final aggregates exist.
    if 'aggregates' not in result:
        continue
    source = Path(result['args']['runs'][0])
    stage = source.parent.name
    checkpoint_seed = source.name.rsplit('_seed', 1)[-1]
    for row in result['aggregates']:
        fields = [stage, checkpoint_seed, row['task'], row['target'], row['arm'],
                  row['method'], str(row['seeds'])]
        fields += [number(row['mean_metrics'][key]) for key in ('mae', 'rmse', 'r2', 'spearman')]
        lines.append('| ' + ' | '.join(str(v).replace('|', '/') for v in fields) + ' |')
    for row in result['records']:
        if 'skipped' in row:
            lines.append(f"\nSkipped {stage}/{source.name}/{row.get('task')}: {row['skipped']}")
lines += ['', 'Fingerprint ridge/RF baselines are repeated per checkpoint invocation; do not count them as independent replicates.',
          'MP uses a finite crystal cluster approximation and has no molecular fingerprint baseline.',
          'See per-unit results.json for split IDs, bootstrap intervals and predictions. Separate arm invocations do not provide cross-arm paired comparisons.']
(SAVE / 'SUMMARY.md').write_text('\n'.join(lines) + '\n')
print('Markdown results:', SAVE / 'SUMMARY.md')

In [ ]:
# Final cell: retain individual summaries and aggregate without including itself.
profile_root = SAVE / 'profiling'
profile_root.mkdir(exist_ok=True)
summaries = {}
for path in sorted(profile_root.rglob('summary.json')):
    if path != profile_root / 'summary.json':
        summaries[str(path.parent.relative_to(profile_root))] = json.loads(path.read_text())
combined = {'runs': summaries, 'run_count': len(summaries),
            'note': 'Individual summaries describe the latest invocation, not cumulative resumed sessions.'}
temporary = profile_root / 'summary.tmp'
temporary.write_text(json.dumps(combined, indent=2))
temporary.replace(profile_root / 'summary.json')
print('Profiling:', profile_root / 'summary.json')